# 📝 Taranga AI: Technical Report — NLP Observation Analysis

**Module:** `03_nlp_analyzer`  
**Objective:** Process qualitative teacher notes using NLP to identify consistent LD indicators.

---

## 1. Methodology: Semantic Tokenization

Teacher observations are inherently unstructured. Taranga uses the **Spacy (en_core_web_sm)** model to perform semantic extraction on text notes. Our pipeline follows these steps:

1. **Normalization:** Convert all text to lowercase and remove punctuation.
2. **Stop-word Filtering:** Filter out insignificant words (the, is, and) to isolate core clinical descriptors.
3. **Lemmatization:** Reduce words to their base form (e.g., "struggled", "struggling", "struggles" all map to "struggle").
4. **Weighted Matching:** Cross-reference lemmas against a dictionary of ~40 clinical indicators.

---

## 2. NLP Pipeline Setup

In [ ]:
import spacy
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import pandas as pd
import seaborn as sns

sns.set_theme(style="white")

try:
    nlp = spacy.load("en_core_web_sm")
except:
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

LD_KEYWORDS = {
    "dyslexia":    ["read", "spell", "slow", "struggle", "reverse", "letter", "word", "aloud"],
    "dyscalculia": ["math", "number", "calculate", "time", "count", "logic", "sequence"],
    "dysgraphia":  ["write", "messy", "grip", "space", "handwriting", "pencil", "draw"],
    "nvld":        ["social", "cue", "routine", "clumsy", "awkward", "rigid", "interaction"],
    "apd":         ["noise", "listen", "hear", "instruction", "focus", "sound", "distract"]
}

## 3. Visualizing Indicator Lexicon

The word cloud below represents the weighted priority of keywords that the NLP engine looks for when processing observation notes.

In [ ]:
# Combine all keywords for a tag cloud
all_text = " ".join([" ".join(v) for v in LD_KEYWORDS.values()])
wordcloud = WordCloud(width=800, height=400, background_color='white', 
                      colormap='magma', max_words=50).generate(all_text)

plt.figure(figsize=(15, 7.5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("NLP Model: Distribution of Diagnostic Indicator Keywords", fontsize=16, fontweight='bold')
plt.show()

## 4. Analysis Report Generation

The system generates a breakdown of detected indicators from a complex sample observation.

In [ ]:
def analyze(text):
    doc = nlp(text.lower())
    lemmas = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    flags = {k: 0 for k in LD_KEYWORDS.keys()}
    for lemma in lemmas:
        for ld, keywords in LD_KEYWORDS.items():
            if lemma in keywords: flags[ld] += 1
    return flags

sample_note = """
Student struggles with basic math and number sequences. 
During reading time, they read aloud very slowly and reversed several letters. 
They also found the classroom background noise very distracting.
"""

results = analyze(sample_note)
res_df = pd.DataFrame(list(results.items()), columns=["Domain", "Matches"])

plt.figure(figsize=(10, 6))
sns.barplot(x="Matches", y="Domain", data=res_df, palette="viridis")
plt.title("NLP Case Analysis: Flagged Indicators from Observation Notes", fontsize=14, fontweight='bold')
plt.xlabel("Count of Clinical Indicator Matches")
plt.show()

print("NLP Summary for Student Profile:")
print("-" * 30)
detected = [k.upper() for k, v in results.items() if v > 0]
print(f"Detected Indicators: {', '.join(detected) if detected else 'None'}")